[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C16_Generative_Models_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与方法论热身

本课全程 **纯 numpy、CPU 可跑**，用 numpy **从零**实现五族生成模型（自写前向/反向/采样），在 **玩具 1D/2D 分布** 上跑到收敛、与解析参考 **对拍**。

这个 notebook 做三件事：① 确认环境；② 用一个最小例子体会「**生成 = 学一个能采样的 p(x)**」与判别的区别；③ 立下全课的纪律——**对拍 / 收敛阈值 + assert**。

## 1 · 环境自检

只需要 `numpy`。`matplotlib` 可选（仅用于画分布散点）。

In [ ]:
import sys, platform
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np
print('numpy', np.__version__)
try:
    import matplotlib; print('matplotlib', matplotlib.__version__, '(可选)')
except Exception:
    print('matplotlib 未安装（可选，不影响课程）')
print('环境就绪 ✅')

## 2 · 判别 vs 生成：一个最小对照

**判别**：给 `x` 猜标签 `y`，建模 `p(y|x)`。**生成**：不给输入，凭空采出新的 `x`，建模 `p(x)`。

我们造一个 1D 双峰分布作为「真实数据」。判别任务是分辨一个点来自左峰还是右峰；生成任务是**采样出新的、像真实数据的点**。先把真实分布建出来。

In [ ]:
rng = np.random.default_rng(0)

def sample_true(n):
    '''真实数据分布：1D 双峰高斯混合（左峰 -2，右峰 +2）。'''
    comp = rng.integers(0, 2, size=n)              # 选哪个峰
    means = np.where(comp == 0, -2.0, 2.0)
    return means + 0.5 * rng.standard_normal(n)

data = sample_true(2000)
print('真实数据：', data.shape, '均值≈%.2f' % data.mean(), '应接近 0（两峰对称）')
assert data.shape == (2000,)
# 双峰：落在 [-1,1] 中间谷底的点应该很少
in_valley = np.mean(np.abs(data) < 1.0)
print('落在中间谷底 |x|<1 的比例 = %.2f （双峰 -> 应较小）' % in_valley)
assert in_valley < 0.35, '双峰之间应是低密度谷'
print('✅ 真实双峰分布构造正确')

## 3 · 判别模型：建模 p(y|x)（容易）

判别只需画一条边界。这里用最朴素的规则 `x>0 -> 右峰`，看它分得多准——**判别不需要理解数据怎么生成**。

In [ ]:
def discriminate(x):
    return (x > 0).astype(int)           # 决策边界：x>0 判为右峰(1)

# 造带标签的测试集
comp = rng.integers(0, 2, size=5000)
means = np.where(comp == 0, -2.0, 2.0)
x_test = means + 0.5 * rng.standard_normal(5000)
pred = discriminate(x_test)
acc = np.mean(pred == comp)
print('判别准确率 = %.3f' % acc)
assert acc > 0.99, '双峰相距很远，一条边界就近乎完美'
print('✅ 判别 p(y|x) 很容易：一条 x>0 的边界就够了。但它【采不出】新样本！')

## 4 · 生成模型：建模 p(x) 并采样（难）

生成要把**整个分布的形状**学下来，才能采出新点。这里我们「作弊」地直接用一个高斯混合当生成模型（真正的模型要从数据里**学**出这些参数——那正是后面五个模块的事），重点是体会：

**采样 = 先选一个隐变量（哪个峰），再从对应分布生成。** 这个「隐变量 → 数据」的两段式，是几乎所有生成模型的范式。

In [ ]:
class ToyGenerator:
    '''玩具生成模型：隐变量 z 选峰，再生成数据。模仿 p(x)=sum_z p(z)p(x|z)。'''
    def __init__(self, means=(-2.0, 2.0), std=0.5):
        self.means = np.array(means); self.std = std
    def sample(self, n):
        z = rng.integers(0, len(self.means), size=n)   # 隐变量：选峰
        return self.means[z] + self.std * rng.standard_normal(n)

gen = ToyGenerator()
fake = gen.sample(2000)
# 生成样本应与真实数据【分布】接近：比较前几阶矩
print('真实  均值=%.2f 标准差=%.2f' % (data.mean(), data.std()))
print('生成  均值=%.2f 标准差=%.2f' % (fake.mean(), fake.std()))
assert abs(fake.mean() - data.mean()) < 0.2
assert abs(fake.std() - data.std()) < 0.2
print('✅ 生成 p(x)：先采隐变量(选峰)再生成 —— 这个两段式贯穿全部五族模型')

## 5 · 立纪律：一个比较两个分布的「对拍」工具

生成模型很难有逐点的 ground truth（采样是随机的）。我们退而求其次：比较两个样本集的**分布**是否接近。

一个简单稳健的标量是 **1D 直方图的 L1 距离**（玩具版的分布距离）。后面判断「生成分布 ≈ 真实分布」就用它。

In [ ]:
def hist_l1(a, b, bins=40, rng_range=(-5, 5)):
    '''两个样本集的归一化直方图之间的 L1 距离，范围 [0,2]，越小越接近。'''
    ha, edges = np.histogram(a, bins=bins, range=rng_range, density=True)
    hb, _ = np.histogram(b, bins=bins, range=rng_range, density=True)
    width = edges[1] - edges[0]
    return float(np.sum(np.abs(ha - hb)) * width)

d_self = hist_l1(data, sample_true(2000))   # 真实 vs 真实（另抽一批）：应很小
d_gen  = hist_l1(data, fake)                # 真实 vs 生成：也应较小
d_bad  = hist_l1(data, rng.standard_normal(2000))  # 真实 vs 单峰标准正态：应较大
print('真实 vs 真实   L1 = %.3f' % d_self)
print('真实 vs 生成   L1 = %.3f' % d_gen)
print('真实 vs 单峰   L1 = %.3f （把双峰错当单峰 -> 明显更大）' % d_bad)
assert d_gen < d_bad, '好的生成模型应比错误模型更接近真实分布'
assert d_self < 0.3
print('\n✅ 这就是全课判分逻辑：生成分布要显著接近真实分布、并优于错误基线。')

## 6 · 一个会贯穿全课的 1D 玩具数据 + 收敛判据

把「玩具数据 + 分布距离阈值」固化成工具。后面每族模型都用类似的判据：固定种子、训练后要求生成分布的距离低于阈值。

In [ ]:
def make_toy_1d(n=2000, seed=0):
    g = np.random.default_rng(seed)
    comp = g.integers(0, 2, size=n)
    return np.where(comp == 0, -2.0, 2.0) + 0.5 * g.standard_normal(n)

def converged(real, gen_samples, thresh=0.4):
    '''判定生成样本是否已收敛到真实分布（直方图 L1 < 阈值）。'''
    d = hist_l1(real, gen_samples)
    print('  分布距离 L1 = %.3f  (阈值 %.2f) -> %s' % (d, thresh, '收敛 ✅' if d < thresh else '未收敛 ❌'))
    return d < thresh

real = make_toy_1d()
ok = converged(real, gen.sample(2000))
assert ok, '玩具生成器应当通过收敛判据'
print('\n这就是全课的工作流：从零实现 -> 在玩具分布上训练 -> 对拍/收敛判据 + assert 兜底。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写出的每族模型，能算解析解就对拍解析解，不能就要求生成分布收敛到阈值；结构正确则数值对上，数值对上则逻辑可迁移到 PyTorch。

**接下来五个模块**：01 自编码器 → 02 VAE → 03 GAN → 04 扩散 DDPM → 05 流匹配。从「只会重建」一步步走到「会采样、会统一」。

下一站：**模块 01 · 自编码器**。